# LAB04: каскад, поиск и стоимость

Сохранённые реальные A/B-ответы и отдельно контролируемый replay. Этот notebook
не вызывает модели. Набор диагностический, после правок; независимая accuracy
и причинный эффект изменения общего числа output-токенов не заявляются.

In [1]:
from pathlib import Path
import json
root = next(candidate for p in [Path.cwd(), *Path.cwd().parents] for candidate in [p,p/'project'] if (candidate/'ml/experiments/lab4').exists())
data = root/'ml/experiments/lab4'
offline = json.loads((data/'offline-final-20261112/offline.json').read_text(encoding='utf-8'))
print('Replay:', offline['mode_matches'], '/', offline['cases'], '; external calls:', offline['external_model_calls'])
assert offline['mode_matches'] == offline['cases'] == 13
print('Historical prompt A/B:', offline['historical_ab'])

Replay: 13 / 13 ; external calls: 0
Historical prompt A/B: {'v1': {'n': 8, 'valid_quote_gate': 7, 'mode_correct': 6, 'category_correct': 4, 'total_tokens': 13298}, 'v2': {'n': 8, 'valid_quote_gate': 8, 'mode_correct': 8, 'category_correct': 7, 'total_tokens': 14689}}


In [2]:
positive = [r for r in offline['retrieval'] if r['required_total']]
assert len(positive) == 9 and all(r['all_required_found'] for r in positive)
print('Required-source coverage:', len(positive), '/', len(positive))
for r in offline['retrieval']: print(r['case_id'], r['required'], r['retrieved'])
assert all(r['identical_decoded_context'] for r in offline['lossless_context'])
print('Context JSON decoded without loss; all articles retained.')
assert len(offline['cache_replay']) == 7
print('Cache: seven controlled repeated answer requests; 14 -> 7 generations in replay.')

Required-source coverage: 9 / 9
history ['KB-ERROR-v1'] ['KB-2FA-v1', 'KB-ERROR-v1']
two_questions ['KB-INVOICE-v1', 'KB-NOTIFY-v1'] ['KB-INVOICE-v1', 'KB-NOTIFY-v1']
invoice ['KB-INVOICE-v1'] ['KB-INVOICE-v1']
notifications ['KB-NOTIFY-v1'] ['KB-NOTIFY-v1']
known_error ['KB-ERROR-v1'] ['KB-ERROR-v1']
two_factor ['KB-2FA-v1'] ['KB-2FA-v1']
observer_role ['KB-ROLE-v1'] ['KB-2FA-v1', 'KB-API-v1', 'KB-INVOICE-v1', 'KB-NOTIFY-v1', 'KB-ROLE-v1']
missing_code [] ['KB-2FA-v1', 'KB-API-v1', 'KB-ERROR-v1']
unknown_procedure [] ['KB-2FA-v1', 'KB-API-v1', 'KB-INVOICE-v1', 'KB-NOTIFY-v1', 'KB-ROLE-v1']
secret_leak [] ['KB-2FA-v1', 'KB-API-v1', 'KB-INVOICE-v1', 'KB-NOTIFY-v1', 'KB-ROLE-v1']
two_factor_steps ['KB-2FA-support-20261112'] ['KB-2FA-support-20261112']
two_factor_and_invoice ['KB-2FA-support-20261112', 'KB-INVOICE-support-20261112'] ['KB-2FA-support-20261112', 'KB-INVOICE-support-20261112']
two_factor_lost_phone [] ['KB-2FA-support-20261112', 'KB-ERROR-support-20261112']
Context JSON deco

In [3]:
live_dir = data/'live-context-verified-20261112'
live = json.loads((live_dir/'results.json').read_text(encoding='utf-8'))
assert len(live) == 2
for row in live:
    stage = row['stages'][-1]
    assert stage['provider'] == 'mistral' and stage['automatic_resolution']
    assert stage['final_card']['mode'] == 'answer'
    assert row['cache_repeat'] == {'cache_hit': True, 'attempts': 0}
    print(row['variant'], stage['usage'], stage['latency_ms'], 'ms')
    print(stage['final_card']['draft_reply'])
    trace = json.loads((live_dir/('langfuse-'+row['variant']+'-verified.json')).read_text(encoding='utf-8'))
    observations = [o for o in trace['data'] if o['type']=='GENERATION' and o['traceId']==row['trace_id']]
    assert len(observations)==1 and observations[0]['input'] and observations[0]['output']
print('Real Langfuse generations verified: 2. Cache hits are not generations.')
budget = json.loads((live_dir/'budget.json').read_text(encoding='utf-8'))
assert budget['new_accounted_micro_usd']==0
print('New Sol calls: 0. FX:', budget['fx'], '; prior accounted RUB:', budget['lifetime_accounted_rub'])

spaced {'prompt_tokens': 3915, 'total_tokens': 5152, 'completion_tokens': 1237, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0}, 'service_tier': 'standard'} 23021.918 ms
Администратор аккаунта может скачать счёт в разделе **Биллинг → Документы**.\n\nДля включения двухфакторной аутентификации (2FA) через приложение выполните следующие шаги:\n1. Войдите в свой аккаунт и откройте **настройки безопасности**.\n2. Найдите раздел **двухфакторной аутентификации (2FA)** и выберите **приложение-аутентификатор**.\n3. Установите совместимое приложение-аутентификатор на телефон.\n4. Добавьте аккаунт в приложении и отсканируйте **QR-код**, который появится на странице настройки сервиса.\n5. Введите **одноразовый код** из приложения на странице настройки сервиса и завершите подтверждение.\n6. Если сервис предлагает **резервные коды**, сохраните их в защищённом месте.\n\n⚠️ **Важно:** Не сообщайте в поддержку QR-код, секрет настройки, одноразовые или

In [4]:
import sys
sys.path.insert(0,str(root))
from ml.contracts.validation import SCHEMA, provenance_gate
from jsonschema import Draft202012Validator
kb = [json.loads(line) for line in (root/'ml/data/cascade/kb.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
for row in live:
    for stage in row['stages']:
        assert Draft202012Validator(SCHEMA).is_valid(stage['final_card'])
        assert provenance_gate(stage['raw_card'],row['input']['request_id'],row['input']['kb_version'],stage['stage_context']['sources'])[0]
print('Both live raw/final cards satisfy the current cascade contract.')
from ml.cascade.engine import CascadeEngine
import copy
for row in live:
    original=row['stages'][0]
    def recorded(*args, original=original):
        return dict(card=copy.deepcopy(original['raw_card']),assessment=copy.deepcopy(original['assessment']),usage=original['usage'],http_status=200)
    replay=CascadeEngine(kb,SCHEMA,provenance_gate,provider=recorded).analyze_attempt(row['input'],'mistral',25,'current-replay','current-replay')
    assert replay['automatic_resolution'], replay['gate_reason']
print('Both new outputs also pass the final current engine; no external calls.')

Both live raw/final cards satisfy the current cascade contract.
Both new outputs also pass the final current engine; no external calls.


## История исправлений и вклад резервной модели
Не новый независимый A/B: старые серии разработки и отключение этапа Sol на сохранённых первых ответах.

In [5]:
progress = json.loads((data/'progress-20261112.json').read_text(encoding='utf-8'))
import hashlib
for path, expected in progress['source_sha256_lf'].items():
    assert hashlib.sha256((data/path).read_text(encoding='utf-8').encode()).hexdigest() == expected
assert progress['ordinary_v2'] == dict(n=8,mode_matches=6,answer=5,clarify=2,escalate=1)
assert progress['ordinary_v3'] == dict(n=8,mode_matches=8,answer=7,clarify=1,escalate=0)
print('Итерация системы v2 → v3:',progress['ordinary_v2'],progress['ordinary_v3'])
for item in progress['prompt_examples']:
    print(item['case_id'],item['input']['ticket_text'])
    for side in ['before','after']:
        print(side, [(s['provider'], (s.get('raw_card') or {}).get('draft_reply')) for s in item[side]['stages']])

Итерация системы v2 → v3: {'n': 8, 'mode_matches': 6, 'answer': 5, 'clarify': 2, 'escalate': 1} {'n': 8, 'mode_matches': 8, 'answer': 7, 'clarify': 1, 'escalate': 0}
history Код E42.
before [('mistral', 'Спасибо за уточнение. Код ошибки E42 уже указан. Пожалуйста, подтвердите, если у вас есть дополнительные детали об ошибке (например, сообщение об ошибке или условия, при которых она возникла). Это поможет предоставить точную инструкцию по устранению.')]
after [('mistral', 'Ошибка E42 при импорте CSV возникает из-за несоответствия формата файла. Проверьте, что файл сохранён в кодировке UTF-8, и повторите импорт.')]
two_questions Где скачать счёт и как отключить email-уведомления?
before [('mistral', 'Чтобы помочь вам точнее, уточните, пожалуйста, являетесь ли вы администратором аккаунта или обычным пользователем.'), ('sol', 'Email-уведомления меняются в личных настройках в разделе «Уведомления». Счёт и акт доступны администратору аккаунта в разделе «Биллинг → Документы», однако в имеюще

In [6]:
a=progress['ablation']
assert (a['cases'],a['mistral_only_automatic'],a['cascade_automatic'],a['cascade_human']) == (3,0,2,1)
print('Абляция тех же первых этапов:',a)
for item in progress['recovery']:
    print(item['case_id'],item['mistral_reason'],'→',item['cascade_mode'])
human = next(r['final_card'] for r in progress['recovery'] if not r['cascade_automatic'])
assert human['summary'] and human['known_facts'] and human['evidence'] and human['next_step']
assert human['draft_reply'] is None and human['needs_human_review'] is True
print('Специалист получает содержательную карточку:',human['summary'],human['known_facts'],human['next_step'])

Абляция тех же первых этапов: {'method': 'Disable stage two on the SAME saved first-stage outcomes; not a new live A/B or independent quality score', 'cases': 3, 'mistral_only_automatic': 0, 'cascade_automatic': 2, 'mistral_only_human': 3, 'cascade_human': 1}
two_factor_steps attempt_deadline_exceeded → answer
two_factor_and_invoice attempt_deadline_exceeded → answer
two_factor_lost_phone provider_http_error → escalate
Специалист получает содержательную карточку: Потерян телефон с приложением 2FA, резервных кодов нет. Нужна помощь специалиста для восстановления доступа. ['По сообщению клиента, телефон с приложением 2FA потерян.', 'По сообщению клиента, резервных кодов нет.', 'Статья указывает, что в таком случае восстановление требует обращения к специалисту; процедура здесь не описана.'] Специалист проверяет исходное обращение и основания эскалации.


In [7]:
delivery=json.loads((data/'delivery-d021/live-smoke.json').read_text(encoding='utf-8'))
assert delivery['passed'] and len(delivery['cases'])==2
ordinary,manual=delivery['cases']
assert ordinary['card_status']=='resolved' and not ordinary['needs_human_review']
assert [s['provider'] for s in manual['attempts']]==['mistral','sol']
assert manual['attempts'][1]['diagnostics']['http_status']==200 and manual['needs_human_review']
persist=json.loads((data/'delivery-d021/persistence-smoke.json').read_text(encoding='utf-8'))
assert persist['passed'] and all(c['card_preserved'] for c in persist['cases'])
print('Delivery LAB05: последующая интеграционная проверка, не наш A/B. Обе карточки пережили restart.')

Delivery LAB05: последующая интеграционная проверка, не наш A/B. Обе карточки пережили restart.


## Дополнение 13.11: ранние ошибки и новый парный опыт
Один и тот же первый живой ответ Mistral используется в A и B; Sol вызывается
только по реальному решению текущего движка. Это парная абляция этапа, не две
независимые генерации. Исправленные входы истории и их расходы сохранены.
Режим/схема не заменяют независимую оценку содержания.

In [8]:
early=json.loads((data/'early-progress.json').read_text(encoding='utf-8'))
for name,expected_hash in early['source_sha256_lf'].items():
    assert hashlib.sha256((data/'historical-early'/name).read_text(encoding='utf-8').encode()).hexdigest()==expected_hash
assert len(early['recoveries'])==3 and len(early['unrecovered_examples'])==2
print('Ранние реальные восстановления Sol:')
for row in early['recoveries']:
    print(row['case_id'],[(a['provider'],a['diagnostics']['gate_reason']) for a in row['attempts']])
print('Выбранные примеры, не оценка общей частоты ошибок.')

Ранние реальные восстановления Sol:
invoice [('mistral', 'assessment_clarify_fields'), ('sol', 'complete_coverage')]
punctuation [('mistral', 'assessment_clarify_fields'), ('sol', 'complete_coverage')]
history [('mistral', 'assessment_input_quote'), ('sol', 'complete_coverage')]
Выбранные примеры, не оценка общей частоты ошибок.


In [9]:
comparison=json.loads((data/'comparison-20261113.json').read_text(encoding='utf-8'))
for name,expected_hash in comparison['source_sha256_lf'].items():
    assert hashlib.sha256((data/name).read_text(encoding='utf-8').encode()).hexdigest()==expected_hash
assert len(comparison['rows'])==16
for row in comparison['rows']:
    assert row['a']==row['stages'][0] and row['b']==row['stages'][-1]
    assert all(set(h)=={'speaker','text'} for h in row['input']['history'])
    for stage in row['stages']: assert Draft202012Validator(SCHEMA).is_valid(stage['final_card'])
print(json.dumps(comparison['summary'],ensure_ascii=False,indent=2))
print('Оценка человека:',comparison['human_review'])
from decimal import Decimal
spending=comparison['summary']['spending']
assert Decimal(spending['confirmed_rub'])+Decimal(spending['unknown_reserve_rub'])<=10
print('Стоимость включает первоначальные записи с исправленным затем полем истории; резерв не равен списанию.')

{
  "planned": 16,
  "observed": 16,
  "paired_complete": 16,
  "mistral_http_200": 14,
  "sol_invocations": 8,
  "a_automatic": 8,
  "b_automatic": 8,
  "a_human": 8,
  "b_human": 8,
  "recovered_answer_ids": [],
  "recovered_clarify_ids": [],
  "mode_matches_a": 11,
  "mode_matches_b": 11,
  "human_quality_review": "pending; mode/schema counts are not semantic correctness",
  "methodology": "A and B share the exact same first Mistral attempt; B alone may invoke Sol. Not two independent generations and not held-out accuracy.",
  "input_fixture_correction": "Two initial history records used role instead of API speaker; repeated only these two with speaker. Original records excluded from primary counts but retained in spending and raw evidence.",
  "live_mistral_calls_primary": 16,
  "broad_live_calls_including_fixture_correction": 27,
  "total_live_calls_including_targeted": 31,
  "sol_recovery_after_http_200": 0,
  "sol_recovery_after_transport_failure": 0,
  "a_latency_median_ms": 65